In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# データを読み込む
train_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")

# 最初の5行だけ中身を見てみる！
train_data.head()

In [ ]:
# 男女別の生存率を計算する
survival_by_gender = train_data.groupby('Sex')['Survived'].mean()
print(survival_by_gender)

In [ ]:
# 客室クラス（Pclass）別の生存率を計算する
survival_by_class = train_data.groupby('Pclass')['Survived'].mean()
print(survival_by_class)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 予測に使うデータ（特徴量）をピックアップ
features = ["Pclass", "Sex", "SibSp", "Parch"]

# AIは文字（male/femaleなど）を読めないので、数字（0と1）に変換する
X = pd.get_dummies(train_data[features])
y = train_data["Survived"]

# AIモデル（ランダムフォレスト）の準備
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)

# AIに過去のデータを読み込ませて「学習」させる！
model.fit(X, y)

print("AIの学習が完了しました！")

In [ ]:
# 1. 本番用（テスト）のデータを読み込む
test_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

# 2. 練習の時と同じ「4つの手がかり」だけを取り出し、数字に翻訳する
X_test = pd.get_dummies(test_data[features])

# 3. 探偵団（AI）に予測させる！
predictions = model.predict(X_test)

# 4. Kaggleに提出するための「解答用紙（CSVファイル）」を作成する
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)

print("提出用のファイル（submission.csv）が完成しました！")

In [ ]:
# データの中に「空欄（NaN）」がいくつあるか数える
print(train_data.isnull().sum())

In [ ]:
# Age（年齢）の空欄を、全体の「中央値」で埋める
train_data['Age'] = train_data['Age'].fillna(train_data['Age'].median())

# もう一度空欄の数をチェック（Ageの空欄が0になっているはず！）
print(train_data.isnull().sum())

In [ ]:
# 「兄弟・配偶者の数（SibSp）」と「親・子供の数（Parch）」に、自分（1）を足して「家族の人数」を作る
train_data['FamilySize'] = train_data['SibSp'] + train_data['Parch'] + 1

# 家族の人数ごとに、生存率がどう変わるか見てみる
survival_by_family = train_data.groupby('FamilySize')['Survived'].mean()
print(survival_by_family)

In [ ]:
# 1. 本番用データ（test_data）にも「全く同じ下ごしらえ」をする
test_data['Age'] = test_data['Age'].fillna(test_data['Age'].median())
test_data['FamilySize'] = test_data['SibSp'] + test_data['Parch'] + 1

# 2. AIに渡す「新しい手がかり」のリスト（AgeとFamilySizeを追加！）
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "FamilySize"]

# 3. データを数字に翻訳する
X_train = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])
y_train = train_data["Survived"]

# 4. 新しい手がかりで探偵団（AI）を再学習させる！
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X_train, y_train)

# 5. 予測して、新しい解答用紙を作る
predictions = model.predict(X_test)
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})

# 今回は「submission_v2.csv」という新しい名前で保存します
output.to_csv('submission.csv', index=False)

print("バージョン2の提出ファイルが完成しました！")
